In [ ]:
# import yfinance as yf
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import yfinance as yf  # Make sure this is installed!

def fetch_hourly_data(ticker):
    try:
        df = yf.download(ticker, period="5d", interval="60m", progress=False)
        return df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    except Exception:
        return pd.DataFrame()

def compute_volatility(df):
    log_returns = np.log(df['Close'] / df['Close'].shift(1)).dropna()
    return log_returns.std()

def markov_predict_trend(df):
    df = df.copy()
    df['Return'] = df['Close'].pct_change()
    df['Trend'] = df['Return'].apply(lambda x: 'Up' if x > 0 else 'Down')
    transitions = pd.crosstab(df['Trend'].shift(), df['Trend'])
    prob_up = transitions.loc['Up', 'Up'] / transitions.loc['Up'].sum() if 'Up' in transitions.index else 0.5
    return 'Up' if prob_up >= 0.5 else 'Down'

def analyze_constituents(file_path):
    constituents_df = pd.read_csv(file_path)
    scores = []
    prices = []

    for _, row in tqdm(constituents_df.iterrows(), total=len(constituents_df)):
        symbol = row['Symbol']
        sector = row['GICS Sub-Industry']
        df = fetch_hourly_data(symbol)

        if df.empty or len(df) < 30:
            scores.append(0)
            prices.append(None)
            continue

        volatility = compute_volatility(df)
        trend = markov_predict_trend(df)
        base_score = int(round(volatility * 10000))
        latest_price = df['Close'].iloc[-1]

        # Sector scoring
        sector_peers = constituents_df[constituents_df['GICS Sub-Industry'] == sector]['Symbol']
        same_sector_risers = 0
        for peer in sector_peers:
            if peer == symbol:
                continue
            peer_df = fetch_hourly_data(peer)
            if peer_df.empty or len(peer_df) < 30:
                continue
            if markov_predict_trend(peer_df) == 'Up':
                same_sector_risers += 1
            if same_sector_risers >= 10:
                break

        sector_score = min(same_sector_risers * 10, 100)
        final_score = sector_score + base_score if trend == 'Up' else sector_score - base_score

        scores.append(final_score)
        prices.append(latest_price)

    constituents_df['Score'] = scores
    constituents_df['Latest Price'] = prices
    return constituents_df

df_result = analyze_constituents(r"C:\TWS API\source\pythonclient\test_data\test_1_data\constituents.csv")
df_result.to_csv("scored_constituents_with_price.csv", index=False)
print(df_result[['Symbol', 'Score', 'Latest Price']].head())

import re

def extract_price(value):
    match = re.search(r'(\d+\.\d+)', str(value))
    return float(match.group(1)) if match else None

df_result['Latest Price'] = df_result['Latest Price'].apply(extract_price)

max_price = float(input("Enter the maximum stock price: "))
filtered = df_result.dropna(subset=['Latest Price'])
filtered = filtered[filtered['Latest Price'] <= max_price]

if not filtered.empty:
    best = filtered.sort_values(by='Score', ascending=False).iloc[0]
    print(f"\nBest stock under ${max_price}:")
    print(f"Symbol: {best['Symbol']}, Score: {best['Score']}, Latest Price: ${best['Latest Price']:.2f}")
else:
    print(f"\nNo stock found under ${max_price}.")
    
folder_name = "screening_reports"
os.makedirs(folder_name, exist_ok=True)

today_str = datetime.now().strftime("%Y-%m-%d")

file_path = os.path.join(folder_name, f"df_result_{today_str}.csv")

df_result.to_csv(file_path, index=False)

print(f"CSV saved to {file_path}")